In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix
from scipy.signal import resample
import time

print("TensorFlow version:", tf.__version__)

# ============================================================
# 1. Download and Extract Original Dataset
# ============================================================
# S1 Dataset from PLOS One (Fang & Shinozaki, 2018)
!wget -O 'pone.0192684.s001.zip' 'https://doi.org/10.1371/journal.pone.0192684.s001'
!unzip -o 'pone.0192684.s001.zip'
# After extraction, the structure should be: ./isolated/001/isolated_strokes/EOG_001_xx_xxx.csv

# ============================================================
# 2. Data Loading Function
# ============================================================
def load_isolated_data(base_dir='./isolated', target_length=1250):
    """
    Load all isolated stroke CSV files.
    Returns: X_horiz, X_vert, X_diff, y (0-indexed labels)
    """
    X_horiz_list, X_vert_list, X_diff_list = [], [], []
    y_list, subject_list = [], []

    # Iterate over 6 participants (001 to 006)
    for subj_id in range(1, 7):
        subj_dir = os.path.join(base_dir, f'{subj_id:03d}', 'isolated_strokes')
        if not os.path.exists(subj_dir):
            print(f"Warning: directory not found: {subj_dir}")
            continue

        csv_files = glob.glob(os.path.join(subj_dir, '*.csv'))

        for fpath in csv_files:
            fname = os.path.basename(fpath)
            # Parse filename: EOG_00x_xx_xxx.csv
            parts = fname.replace('.csv', '').split('_')
            # parts = ['EOG', '001', '01', '001']
            stroke_id = int(parts[3])  # last part is stroke index (1-12)

            if stroke_id < 1 or stroke_id > 12:
                continue  # Skip invalid strokes

            # Read CSV: column 0 = vertical, column 1 = horizontal (per README)
            df = pd.read_csv(fpath, header=None)
            if df.shape[1] < 2 or df.shape[0] < 10:
                continue

            v_signal = df.iloc[:, 0].values.astype(float)
            h_signal = df.iloc[:, 1].values.astype(float)

            # Resample to fixed length (1250 samples)
            h_resampled = resample(h_signal, target_length)
            v_resampled = resample(v_signal, target_length)
            d_resampled = h_resampled - v_resampled  # Difference channel

            X_horiz_list.append(h_resampled)
            X_vert_list.append(v_resampled)
            X_diff_list.append(d_resampled)
            y_list.append(stroke_id - 1)  # Convert to 0-indexed (0-11)
            subject_list.append(subj_id)

    X_horiz = np.array(X_horiz_list)
    X_vert = np.array(X_vert_list)
    X_diff = np.array(X_diff_list)
    y = np.array(y_list)
    subjects = np.array(subject_list)

    print(f"Total samples loaded: {len(y)}")
    print(f"Samples per subject: {[np.sum(subjects==s) for s in range(1,7)]}")

    return X_horiz, X_vert, X_diff, y, subjects

# Load the data
X_horiz, X_vert, X_diff, y, subjects = load_isolated_data()

# Reshape for Conv1D: (samples, timesteps, channels)
X_horiz = X_horiz.reshape(X_horiz.shape[0], X_horiz.shape[1], 1)
X_vert = X_vert.reshape(X_vert.shape[0], X_vert.shape[1], 1)
X_diff = X_diff.reshape(X_diff.shape[0], X_diff.shape[1], 1)
y_cat = to_categorical(y, num_classes=12)

print(f"X_horiz shape: {X_horiz.shape}")
print(f"y shape: {y.shape}, classes: {np.unique(y)}")

# ============================================================
# 3. Model Architecture 
# ============================================================
def create_1d_cnn(input_shape=(1250, 1)):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=(1250, 1)), 
        MaxPooling1D(pool_size=2), 
        Conv1D(64, 3, activation='relu'), 
        GlobalAveragePooling1D(), 
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(12, activation='softmax')

    ])
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ============================================================
# 4. LOSO Evaluation Loop
# ============================================================
unique_subjects = np.unique(subjects)
n_subjects = len(unique_subjects)

# Storage for results
loso_results = []
all_cms_part1, all_cms_part2, all_cms_part3, all_cms_final = [], [], [], []

total_epochs = 600
batch_size = 32

print(f"\n{'='*60}")
print(f"Starting LOSO Evaluation: {n_subjects} folds (one per subject)")
print(f"{'='*60}")

for fold_idx, test_subj in enumerate(unique_subjects):
    print(f"\n=== Fold {fold_idx+1}/{n_subjects}: "
          f"Test Subject = {test_subj} ===")

    # Split data: test on one subject, train on the other 5
    train_mask = subjects != test_subj
    test_mask = subjects == test_subj

    X_h_train, X_h_test = X_horiz[train_mask], X_horiz[test_mask]
    X_v_train, X_v_test = X_vert[train_mask], X_vert[test_mask]
    X_d_train, X_d_test = X_diff[train_mask], X_diff[test_mask]

    y_train_cat, y_test_cat = y_cat[train_mask], y_cat[test_mask]
    y_test_idx = np.argmax(y_test_cat, axis=1)

    print(f"Train samples: {len(X_h_train)}, Test samples: {len(X_h_test)}")

    # Create fresh models for each fold
    part1_model = create_1d_cnn()
    part2_model = create_1d_cnn()
    part3_model = create_1d_cnn()

    # Train the three branches
    print("Training Part 1 (Horizontal)...")
    part1_model.fit(X_h_train, y_train_cat, epochs=total_epochs,
                    batch_size=batch_size, verbose=0, shuffle=True)

    print("Training Part 2 (Vertical)...")
    part2_model.fit(X_v_train, y_train_cat, epochs=total_epochs,
                    batch_size=batch_size, verbose=0, shuffle=True)

    print("Training Part 3 (Difference)...")
    part3_model.fit(X_d_train, y_train_cat, epochs=total_epochs,
                    batch_size=batch_size, verbose=0, shuffle=True)

    # Freeze branch weights
    part1_model.trainable = False
    part2_model.trainable = False
    part3_model.trainable = False

    # Build and train fusion classifier
    p1_train = part1_model.predict(X_h_train, verbose=0)
    p2_train = part2_model.predict(X_v_train, verbose=0)
    p3_train = part3_model.predict(X_d_train, verbose=0)
    combined_train = np.concatenate([p1_train, p2_train, p3_train], axis=1)

    fusion_model = Sequential([
        Dense(16, activation='relu', input_shape=(36,)),
        Dense(12, activation='softmax')
    ])
    fusion_model.compile(optimizer='adam', loss='categorical_crossentropy',
                         metrics=['accuracy'])

    print("Training Fusion Classifier...")
    fusion_model.fit(combined_train, y_train_cat, epochs=total_epochs,
                     batch_size=batch_size, verbose=0, shuffle=True)

    # Evaluation on test subject
    p1_test = part1_model.predict(X_h_test, verbose=0)
    p2_test = part2_model.predict(X_v_test, verbose=0)
    p3_test = part3_model.predict(X_d_test, verbose=0)
    combined_test = np.concatenate([p1_test, p2_test, p3_test], axis=1)

    p1_pred = np.argmax(p1_test, axis=1)
    p2_pred = np.argmax(p2_test, axis=1)
    p3_pred = np.argmax(p3_test, axis=1)
    final_pred = np.argmax(fusion_model.predict(combined_test, verbose=0), axis=1)

    # Compute confusion matrices
    cm1 = confusion_matrix(y_test_idx, p1_pred, labels=range(12))
    cm2 = confusion_matrix(y_test_idx, p2_pred, labels=range(12))
    cm3 = confusion_matrix(y_test_idx, p3_pred, labels=range(12))
    cm_final = confusion_matrix(y_test_idx, final_pred, labels=range(12))

    all_cms_part1.append(cm1)
    all_cms_part2.append(cm2)
    all_cms_part3.append(cm3)
    all_cms_final.append(cm_final)

    acc1 = np.trace(cm1) / np.sum(cm1)
    acc2 = np.trace(cm2) / np.sum(cm2)
    acc3 = np.trace(cm3) / np.sum(cm3)
    acc_final = np.trace(cm_final) / np.sum(cm_final)

    print(f"  Subject {test_subj} - Part1: {acc1:.4f}, "
          f"Part2: {acc2:.4f}, Part3: {acc3:.4f}, "
          f"Final: {acc_final:.4f}")

    loso_results.append({
        'test_subject': test_subj,
        'acc_part1': acc1, 'acc_part2': acc2,
        'acc_part3': acc3, 'acc_final': acc_final,
        'n_test': len(y_test_idx)
    })

# ============================================================
# 5. Aggregate LOSO Results
# ============================================================
print(f"\n{'='*60}")
print("LOSO Aggregated Results (weighted by test set size)")
print(f"{'='*60}")

df_results = pd.DataFrame(loso_results)

# Weighted average (more fair since subjects have different sample counts)
total_test_samples = df_results['n_test'].sum()
weighted_acc1 = np.sum(df_results['acc_part1'] * df_results['n_test']) / total_test_samples
weighted_acc2 = np.sum(df_results['acc_part2'] * df_results['n_test']) / total_test_samples
weighted_acc3 = np.sum(df_results['acc_part3'] * df_results['n_test']) / total_test_samples
weighted_final = np.sum(df_results['acc_final'] * df_results['n_test']) / total_test_samples

# Simple mean (each subject counts equally)
mean_acc1 = df_results['acc_part1'].mean()
mean_acc2 = df_results['acc_part2'].mean()
mean_acc3 = df_results['acc_part3'].mean()
mean_final = df_results['acc_final'].mean()

print(f"\n{'Branch':<15} {'Weighted Acc':<15} {'Mean Acc':<15} {'Std Dev':<15}")
print("-" * 60)
print(f"{'Part1 (Horiz)':<15} {weighted_acc1:<15.4f} {mean_acc1:<15.4f} {df_results['acc_part1'].std():<15.4f}")
print(f"{'Part2 (Vert)':<15} {weighted_acc2:<15.4f} {mean_acc2:<15.4f} {df_results['acc_part2'].std():<15.4f}")
print(f"{'Part3 (Diff)':<15} {weighted_acc3:<15.4f} {mean_acc3:<15.4f} {df_results['acc_part3'].std():<15.4f}")
print(f"{'FUSED (Final)':<15} {weighted_final:<15.4f} {mean_final:<15.4f} {df_results['acc_final'].std():<15.4f}")

# Aggregate confusion matrix (summed across folds)
agg_cm_final = np.sum(all_cms_final, axis=0)
overall_acc = np.trace(agg_cm_final) / np.sum(agg_cm_final)
print(f"\nOverall LOSO Accuracy (aggregated CM): {overall_acc:.4f}")

# Save results
np.save('loso_confusion_matrices_final.npy', np.array(all_cms_final))
np.save('loso_aggregated_cm.npy', agg_cm_final)
df_results.to_csv('loso_results_per_subject.csv', index=False)
print("\nResults saved to: loso_results_per_subject.csv")



TensorFlow version: 2.20.0
--2026-09-08 18:34:49--  https://doi.org/10.1371/journal.pone.0192684.s001
Resolving doi.org (doi.org)... 172.67.69.3, 104.26.5.132, 104.26.4.132, ...
Connecting to doi.org (doi.org)|172.67.69.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dx.plos.org/10.1371/journal.pone.0192684.s001 [following]
--2026-09-08 18:34:49--  https://dx.plos.org/10.1371/journal.pone.0192684.s001
Resolving dx.plos.org (dx.plos.org)... 35.190.43.188
Connecting to dx.plos.org (dx.plos.org)|35.190.43.188|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://journals.plos.org/plosone/doi?id=10.1371/journal.pone.0192684.s001 [following]
--2026-09-08 18:34:49--  https://journals.plos.org/plosone/doi?id=10.1371/journal.pone.0192684.s001
Resolving journals.plos.org (journals.plos.org)... 35.190.43.188
Connecting to journals.plos.org (journals.plos.org)|35.190.43.188|:443... connected.
HTTP request sent, 

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Part 1 (Horizontal)...
Training Part 2 (Vertical)...
Training Part 3 (Difference)...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Fusion Classifier...
  Subject 1 - Part1: 0.5417, Part2: 0.4917, Part3: 0.0833, Final: 0.8417

=== Fold 2/6: Test Subject = 2 ===
Train samples: 604, Test samples: 120
Training Part 1 (Horizontal)...


...

Training Fusion Classifier...
  Subject 5 - Part1: 0.5167, Part2: 0.3833, Part3: 0.2417, Final: 0.6917

=== Fold 6/6: Test Subject = 6 ===
Train samples: 603, Test samples: 121
Training Part 1 (Horizontal)...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Part 2 (Vertical)...
Training Part 3 (Difference)...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Fusion Classifier...
  Subject 6 - Part1: 0.4545, Part2: 0.5702, Part3: 0.4959, Final: 0.8512

LOSO Aggregated Results (weighted by test set size)

Branch          Weighted Acc    Mean Acc        Std Dev        
------------------------------------------------------------
Part1 (Horiz)   0.5470          0.5469          0.0581         
Part2 (Vert)    0.4586          0.4589          0.1158         
Part3 (Diff)    0.4144          0.4140          0.2215         
FUSED (Final)   0.7914          0.7917          0.1129         

Overall LOSO Accuracy (aggregated CM): 0.7914

Results saved to: loso_results_per_subject.csv

COMPARISON: Subject-Mixed vs LOSO
Subject-Mixed (5-fold): 0.9254
LOSO (6-fold):          0.7914
Drop in accuracy:       0.1340 (13.40 pp)


In [3]:
# ============================================================
# 6. Compare with Subject-Mixed Results
# ============================================================
print(f"\n{'='*60}")
print("COMPARISON: Subject-Mixed vs LOSO")
print(f"{'='*60}")
subject_mixed_acc = 0.8536  
print(f"Subject-Mixed (5-fold): {subject_mixed_acc:.4f}")
print(f"LOSO (6-fold):          {overall_acc:.4f}")
print(f"Drop in accuracy:       {subject_mixed_acc - overall_acc:.4f} "
      f"({(subject_mixed_acc - overall_acc)*100:.2f} pp)")


COMPARISON: Subject-Mixed vs LOSO
Subject-Mixed (5-fold): 0.8536
LOSO (6-fold):          0.7914
Drop in accuracy:       0.0622 (6.22 pp)


In [4]:

# ============================================================
# 1. Mean Confusion Matrices (LOSO)
# ============================================================
mean_cm_part1_loso = np.mean(all_cms_part1, axis=0)
mean_cm_part2_loso = np.mean(all_cms_part2, axis=0)
mean_cm_part3_loso = np.mean(all_cms_part3, axis=0)
mean_cm_final_loso = np.mean(all_cms_final, axis=0)

print("\n" + "="*60)
print("MEAN CONFUSION MATRICES (LOSO)")
print("="*60)
print("\nMean Confusion Matrix - Part1 (LOSO):\n", mean_cm_part1_loso)
print("\nMean Confusion Matrix - Part2 (LOSO):\n", mean_cm_part2_loso)
print("\nMean Confusion Matrix - Part3 (LOSO):\n", mean_cm_part3_loso)
print("\nMean Confusion Matrix - Final Model (LOSO):\n", mean_cm_final_loso)

# Save LOSO mean confusion matrices
np.save('loso_mean_cm_part1.npy', mean_cm_part1_loso)
np.save('loso_mean_cm_part2.npy', mean_cm_part2_loso)
np.save('loso_mean_cm_part3.npy', mean_cm_part3_loso)
np.save('loso_mean_cm_final.npy', mean_cm_final_loso)
print("\n✓ LOSO mean confusion matrices saved.")

# ============================================================
# 2. Calculate Metrics Function 
# ============================================================
def calculate_metrics_loso(cm, model_name):
    num_classes = cm.shape[0]
    precision = []
    recall = []
    f1 = []

    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_scr = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

        precision.append(prec)
        recall.append(rec)
        f1.append(f1_scr)

    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    accuracy = np.trace(cm) / np.sum(cm)

    print(f"\n--- {model_name} Metrics (LOSO) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall: {macro_recall:.4f}")
    print(f"Macro F1-score: {macro_f1:.4f}")

    return accuracy, macro_precision, macro_recall, macro_f1

# ============================================================
# 3. Print Metrics for All Branches (LOSO)
# ============================================================
print("\n" + "="*60)
print("METRICS PER BRANCH (LOSO)")
print("="*60)

acc_p1, prec_p1, rec_p1, f1_p1 = calculate_metrics_loso(mean_cm_part1_loso, "Part1 (Horizontal) - LOSO")
acc_p2, prec_p2, rec_p2, f1_p2 = calculate_metrics_loso(mean_cm_part2_loso, "Part2 (Vertical) - LOSO")
acc_p3, prec_p3, rec_p3, f1_p3 = calculate_metrics_loso(mean_cm_part3_loso, "Part3 (Difference) - LOSO")
acc_fin, prec_fin, rec_fin, f1_fin = calculate_metrics_loso(mean_cm_final_loso, "Final (Fused) - LOSO")

# ============================================================
# 4. Accuracy Per Fold (= Per Subject in LOSO)
# ============================================================
print("\n" + "="*60)
print("ACCURACY PER SUBJECT (= FOLD IN LOSO)")
print("="*60)

print(df_results.to_string(index=False))

# ============================================================
# 5. Mean and Variance of Accuracies across Subjects
# ============================================================
print("\n" + "="*60)
print("MEAN & VARIANCE OF ACCURACIES (across 6 subjects)")
print("="*60)

print(f"Part1 (Horiz.)  - Mean: {df_results['acc_part1'].mean():.4f}, "
      f"Variance: {df_results['acc_part1'].var():.4f}, "
      f"Std: {df_results['acc_part1'].std():.4f}")
print(f"Part2 (Vert.)   - Mean: {df_results['acc_part2'].mean():.4f}, "
      f"Variance: {df_results['acc_part2'].var():.4f}, "
      f"Std: {df_results['acc_part2'].std():.4f}")
print(f"Part3 (Diff.)   - Mean: {df_results['acc_part3'].mean():.4f}, "
      f"Variance: {df_results['acc_part3'].var():.4f}, "
      f"Std: {df_results['acc_part3'].std():.4f}")
print(f"Final (Fused)   - Mean: {df_results['acc_final'].mean():.4f}, "
      f"Variance: {df_results['acc_final'].var():.4f}, "
      f"Std: {df_results['acc_final'].std():.4f}")




# ============================================================
# 9. Final Summary Table 
# ============================================================
print("\n" + "="*60)
print("FINAL SUMMARY - LOSO EVALUATION")
print("="*60)
print(f"{'Branch':<15} {'Accuracy':<12} {'Macro P':<12} {'Macro R':<12} {'Macro F1':<12}")
print("-" * 63)
print(f"{'Part1 (Horiz)':<15} {acc_p1:<12.4f} {prec_p1:<12.4f} {rec_p1:<12.4f} {f1_p1:<12.4f}")
print(f"{'Part2 (Vert)':<15} {acc_p2:<12.4f} {prec_p2:<12.4f} {rec_p2:<12.4f} {f1_p2:<12.4f}")
print(f"{'Part3 (Diff)':<15} {acc_p3:<12.4f} {prec_p3:<12.4f} {rec_p3:<12.4f} {f1_p3:<12.4f}")
print(f"{'FUSED':<15} {acc_fin:<12.4f} {prec_fin:<12.4f} {rec_fin:<12.4f} {f1_fin:<12.4f}")

# ============================================================
# 10. Save All Results
# ============================================================
summary = {
    'Protocol': 'LOSO (6-fold, 6 subjects)',
    'Part1_Accuracy': acc_p1, 'Part1_Precision': prec_p1,
    'Part1_Recall': rec_p1, 'Part1_F1': f1_p1,
    'Part2_Accuracy': acc_p2, 'Part2_Precision': prec_p2,
    'Part2_Recall': rec_p2, 'Part2_F1': f1_p2,
    'Part3_Accuracy': acc_p3, 'Part3_Precision': prec_p3,
    'Part3_Recall': rec_p3, 'Part3_F1': f1_p3,
    'Final_Accuracy': acc_fin, 'Final_Precision': prec_fin,
    'Final_Recall': rec_fin, 'Final_F1': f1_fin,
}

import json
with open('loso_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✓ All LOSO results saved to 'loso_summary.json'")
print("✓ Confusion matrices saved: loso_mean_cm_*.npy")
print("✓ Per-subject results saved: loso_results_per_subject.csv")

# ============================================================
# 11. Comparison with Subject-Mixed
# ============================================================
print("\n" + "="*60)
print("COMPARISON WITH SUBJECT-MIXED RESULTS")
print("="*60)

subject_mixed_results = {
    'Part1': 0.5953, 'Part2': 0.3577, 'Part3': 0.4669, 'Final': 0.8536
}

print(f"{'Branch':<15} {'Subject-Mixed':<15} {'LOSO':<15} {'Δ (pp)':<10}")
print("-" * 55)
print(f"{'Part1 (Horiz)':<15} {subject_mixed_results['Part1']:<15.4f} {acc_p1:<15.4f} {(subject_mixed_results['Part1']-acc_p1)*100:<10.2f}")
print(f"{'Part2 (Vert)':<15} {subject_mixed_results['Part2']:<15.4f} {acc_p2:<15.4f} {(subject_mixed_results['Part2']-acc_p2)*100:<10.2f}")
print(f"{'Part3 (Diff)':<15} {subject_mixed_results['Part3']:<15.4f} {acc_p3:<15.4f} {(subject_mixed_results['Part3']-acc_p3)*100:<10.2f}")
print(f"{'FUSED':<15} {subject_mixed_results['Final']:<15.4f} {acc_fin:<15.4f} {(subject_mixed_results['Final']-acc_fin)*100:<10.2f}")


MEAN CONFUSION MATRICES (LOSO)

Mean Confusion Matrix - Part1 (LOSO):
 [[5.5        0.16666667 0.16666667 0.16666667 1.83333333 0.16666667
  0.         0.16666667 0.         1.83333333 0.         0.        ]
 [0.         5.         1.16666667 0.33333333 0.         0.
  0.         0.         0.83333333 0.         0.         2.66666667]
 [0.         0.33333333 9.16666667 0.         0.         0.
  0.         0.         0.16666667 0.         0.         0.33333333]
 [3.83333333 0.66666667 0.16666667 0.33333333 3.         0.16666667
  0.         0.         0.16666667 1.66666667 0.         0.        ]
 [2.         0.33333333 0.16666667 1.5        4.         0.16666667
  0.         0.5        0.16666667 1.33333333 0.         0.        ]
 [0.16666667 0.16666667 0.         0.         0.5        8.33333333
  0.33333333 0.16666667 0.         0.5        0.         0.        ]
 [0.         0.         0.         0.         0.         0.16666667
  8.16666667 0.         0.         0.         1.833333